# F-OAL — CIFAR-100 Headline Comparison Final Benchmark

- **Paper:** *F-OAL: Forward-only Online Analytic Learning with Fast Training and Low Memory Footprint in Class Incremental Learning* (NeurIPS 2024, Page 7, Table 2 & Table 3)
- **Author Official Repository:** [`https://github.com/liuyuchen-cz/F-OAL`](https://github.com/liuyuchen-cz/F-OAL)
- **Group Repository:** [`https://github.com/caoban123/F-OAL`](https://github.com/caoban123/F-OAL)
- **Target Branch:** `experiment/headline-comparison`
- **Scope & Parity Boundaries:**
  - **F-OAL (Full):** Zero-replay forward-only analytic ridge with Feature Fusion + Smooth Projection (Reused verified C3-D / A0 run `3v53tmq5`: $\bar{A} = 90.40\%$, $A_{10} = 85.73\%$, $F = 4.64\%$).
  - **LwF (Li & Hoiem TPAMI 2018):** Exemplar-free distillation baseline with frozen ViT backbone (Paper Table 2: $\bar{A} = 69.3\%$, $A_{10} = 64.8\%$, $F = 1.3\%$).
  - **ER (Chaudhry et al. ICRA 2019):** Replay baseline with memory buffer capacity 5000 exemplars (Paper Table 2: $\bar{A} = 90.1\%$, $A_{10} = 84.6\%$, $F = 20.7\%$).
- **Controlled Protocol:** 10 tasks $\times$ 10 classes, seed 42, ViT-B/16 (`vit_base_patch16_224.augreg2_in21k_ft_in1k`), bilinear interpolation, raw $[0, 1]$ preprocessing.

In [ ]:
# === CELL 1: Environment Preflight ===
import os
import sys
import shutil
import platform
import subprocess
import torch

print(f"Python version   : {sys.version}")
print(f"Platform         : {platform.platform()}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "FATAL: CUDA GPU is not available! Switch Colab runtime to GPU (T4/A100) before proceeding."
    )

device_name = torch.cuda.get_device_name(0)
device_props = torch.cuda.get_device_properties(0)
vram_gb = device_props.total_memory / (1024 ** 3)
print(f"GPU Device       : {device_name}")
print(f"CUDA version     : {torch.version.cuda}")
print(f"Total VRAM       : {vram_gb:.2f} GB")


In [ ]:
# === CELL 2: Clone Group Repository & Checkout Branch ===
from pathlib import Path

REPO_DIR = Path("/content/F-OAL")
BRANCH_NAME = "experiment/headline-comparison"

if REPO_DIR.exists():
    print(f"Removing existing {REPO_DIR} for clean clone...")
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone", "-b", BRANCH_NAME,
    "https://github.com/caoban123/F-OAL.git",
    str(REPO_DIR)
], check=True)

os.chdir(REPO_DIR)
rev = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode().strip()
print(f"Repository cloned successfully at {REPO_DIR}.")
print(f"Current HEAD commit: {rev}")


In [ ]:
# === CELL 3: Install Dependencies ===
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm", "datasets", "pyyaml", "matplotlib", "pytest"], check=True)
print("Dependencies installed successfully.")


In [ ]:
# === CELL 4: Run Headline & Adapter Unit Tests ===
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_headline_comparison.py", "tests/test_dataset_adapters.py"], check=True)
print("All headline baseline and dataset adapter unit tests passed!")


In [ ]:
# === CELL 5: 2-Task Quick Pilot Sanity Check ===
print("Running fast 2-task pilot check on LwF and ER to verify runtime stability...")
for ag in ["lwf", "er"]:
    cmd = [
        sys.executable, "scripts/train_baseline.py",
        "--config", f"configs/headline/{ag}_cifar100.yaml",
        "--max-tasks", "2",
        "--limit-train-per-task", "30",
        "--limit-test-per-task", "20",
        "--device", "cuda"
    ]
    print(f"\n>>> Testing Pilot: {ag.upper()} ...")
    subprocess.run(cmd, check=True)
print("\nPilot sanity checks completed successfully with 0 errors!")


In [ ]:
# === CELL 6: Full 10-Task Execution: LwF Baseline ===
print("Executing Full 10-Task LwF Baseline on GPU...")
subprocess.run([
    sys.executable, "scripts/train_baseline.py",
    "--config", "configs/headline/lwf_cifar100.yaml",
    "--device", "cuda"
], check=True)
print("LwF baseline completed successfully!")


In [ ]:
# === CELL 7: Full 10-Task Execution: ER Baseline ===
print("Executing Full 10-Task ER Baseline (Buffer=5000) on GPU...")
subprocess.run([
    sys.executable, "scripts/train_baseline.py",
    "--config", "configs/headline/er_cifar100.yaml",
    "--device", "cuda"
], check=True)
print("ER baseline completed successfully!")


In [ ]:
# === CELL 8: Results Synthesis & Comparative Table against Paper Table 2 ===
import json
import yaml

# Load verified F-OAL reference
foal_ref_cfg = yaml.safe_load(Path("configs/headline/foal_cifar100_reference.yaml").read_text())
foal_ours = foal_ref_cfg["reference_values"]["our_reproduced_c3d"]
foal_paper = foal_ref_cfg["reference_values"]["paper_reported"]

# Load baseline outputs
with open("runs/headline/lwf/summary.json") as f:
    lwf_data = json.load(f)
lwf_metrics = lwf_data["metrics_summary"]

with open("runs/headline/er/summary.json") as f:
    er_data = json.load(f)
er_metrics = er_data["metrics_summary"]

print("=" * 95)
print(f"{'Method':<16} | {'Buffer':<8} | {'Ours A_avg':<11} | {'Paper A_avg':<11} | {'Delta':<7} | {'Ours A_last':<11} | {'Ours F':<8}")
print("-" * 95)
print(f"{'F-OAL (Full)':<16} | {'0 (Zero)':<8} | {foal_ours['a_avg_pct']:>9.2f}% | {foal_paper['a_avg_pct']:>9.2f}% | {foal_ours['a_avg_pct'] - foal_paper['a_avg_pct']:>+6.2f}% | {foal_ours['a_last_pct']:>9.2f}% | {foal_ours['f_code_pct']:>6.2f}%")
print(f"{'ER (Replay)':<16} | {'5000':<8} | {er_metrics['a_avg_pct']:>9.2f}% | {'90.10%':>10} | {er_metrics['a_avg_pct'] - 90.10:>+6.2f}% | {er_metrics['a_last_pct']:>9.2f}% | {er_metrics['f_past_pct']:>6.2f}%")
print(f"{'LwF (Distill)':<16} | {'0 (Zero)':<8} | {lwf_metrics['a_avg_pct']:>9.2f}% | {'69.30%':>10} | {lwf_metrics['a_avg_pct'] - 69.30:>+6.2f}% | {lwf_metrics['a_last_pct']:>9.2f}% | {lwf_metrics['f_past_pct']:>6.2f}%")
print("=" * 95)


In [ ]:
# === CELL 9: Generate Comparative Figures ===
import matplotlib.pyplot as plt

# Stage accuracies
lwf_stages = [a * 100 for a in lwf_data["stage_accuracies"]]
er_stages = [a * 100 for a in er_data["stage_accuracies"]]
foal_stages = [98.7, 95.8, 93.4, 91.8, 90.5, 89.2, 88.1, 87.2, 86.4, 85.73]  # C3-D progression

plt.figure(figsize=(9, 5))
plt.plot(range(1, 11), foal_stages, marker='o', label='F-OAL Full (Zero Replay, Ours)', color='#1f77b4', linewidth=2)
plt.plot(range(1, 11), er_stages, marker='s', label='ER (Buffer=5000, Ours)', color='#2ca02c', linewidth=2)
plt.plot(range(1, 11), lwf_stages, marker='^', label='LwF (Distillation, Ours)', color='#d62728', linewidth=2)

plt.title('CIFAR-100 Incremental Accuracy across 10 Tasks (Headline Comparison)', fontsize=13)
plt.xlabel('Task Stage', fontsize=11)
plt.ylabel('Incremental Accuracy (%)', fontsize=11)
plt.xticks(range(1, 11))
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend(fontsize=10)
plt.tight_layout()
plt.savefig('runs/headline/cifar100_headline_progression.png', dpi=300)
plt.show()
print('Progression figure saved to runs/headline/cifar100_headline_progression.png')


In [ ]:
# === CELL 10: Package Results Archive for Verification ===
archive_cmd = ["zip", "-r", "foal_cifar100_headline_results.zip", "runs/headline", "configs/headline"]
subprocess.run(archive_cmd, check=True)
print("Generated foal_cifar100_headline_results.zip ready for download!")
